# 03 - Train Random Forest Stress-Event Classifier

**ITAI 2376 Final Project - GridGuard-AI**  
**Authors:** DeMarcus Crump & Yoana Cook

Trains a Random Forest classifier that predicts `is_stress_event` (the engineered stress flag in `gridguard.db`) from 39 tabular features. The Grid Monitor agent loads this model at runtime as a deterministic second opinion alongside the LSTM forecast.

### Blueprint-locked specification
- **Features:** 39 engineered predictors covering load, reserves, weather, temporal, lag, and rolling-statistic families
- **Target:** `is_stress_event` (engineered during DB construction)
- **Target metrics:** accuracy >= 95 %, recall on stress events >= 85 % on a held-out temporal tail
- **Output:** `models/random_forest.pkl`  +  `models/rf_feature_names.json`  +  `models/rf_config.json`

### Deep-learning / ML connection
Random Forest was chosen over a deeper network because grid operators need **interpretable** alerts - feature-importance scores tell you *why* the model flagged an event. That is a direct application of the tree-ensemble and feature-engineering concepts covered in the course.

Expected runtime in Colab: ~1 min.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install pandas scikit-learn matplotlib joblib
    REPO_URL = os.environ.get('GRIDGUARD_REPO_URL', 'https://github.com/<YOUR_GITHUB_USER>/Crump_GridGuard_ITAI2376.git')
    if not os.path.isdir('/content/repo'):
        !git clone {REPO_URL} /content/repo
    os.chdir('/content/repo')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import json, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support

np.random.seed(42)

In [ ]:
# --- 39 engineered features (blueprint-locked) ---
FEATURES = [
    # Core load + reserves
    'load_mw', 'estimated_capacity', 'reserve_margin', 'reserve_margin_mw', 'stress_index',
    'load_deviation_from_hourly_avg', 'load_deviation_pct',
    # Weather (statewide aggregates)
    'temp_avg_statewide', 'temp_min_statewide', 'temp_max_statewide',
    'wind_avg_statewide', 'wind_max_statewide', 'wind_chill_statewide', 'humidity_avg_statewide',
    'temp_ramp_1h', 'temp_ramp_6h', 'temp_ramp_24h',
    'is_extreme_cold', 'is_extreme_heat',
    # Temporal encodings
    'hour', 'day_of_week', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    # Lag features
    'load_lag_1', 'load_lag_3', 'load_lag_6', 'load_lag_24', 'load_lag_168',
    # Rolling stats
    'load_rolling_mean_3h', 'load_rolling_std_3h',
    'load_rolling_mean_24h', 'load_rolling_std_24h',
    # Ramp / rate-of-change
    'load_ramp_1h', 'load_ramp_6h', 'load_pct_change_24h',
]
TARGET = 'is_stress_event'
assert len(FEATURES) == 39, f'Expected 39 features, got {len(FEATURES)}'

conn = sqlite3.connect('data/gridguard.db')
df = pd.read_sql(
    f"SELECT timestamp, {', '.join(FEATURES)}, {TARGET} FROM ercot_telemetry ORDER BY timestamp",
    conn, parse_dates=['timestamp']
).dropna()
conn.close()

print(f'Total rows after dropna: {len(df):,}')
print(f'Positive class rate    : {df[TARGET].mean()*100:.2f} %')
print(f'Date range             : {df["timestamp"].min()} -> {df["timestamp"].max()}')

In [ ]:
# --- 80 / 20 temporal split (no shuffling - the tail is unseen future) ---
split = int(len(df) * 0.80)
train, test = df.iloc[:split], df.iloc[split:]

X_train, y_train = train[FEATURES].values, train[TARGET].values.astype(int)
X_test,  y_test  = test[FEATURES].values,  test[TARGET].values.astype(int)

print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')
print(f'Train positive rate : {y_train.mean()*100:.2f} %')
print(f'Test positive rate  : {y_test.mean()*100:.2f} %')

In [ ]:
# --- Train ---
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',   # stress events are the minority class
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print('Training complete.')

In [ ]:
# --- Evaluate ---
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
accuracy = (y_pred == y_test).mean()
auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else float('nan')

print(f'Accuracy       : {accuracy*100:.2f} %')
print(f'Precision      : {precision*100:.2f} %')
print(f'Recall         : {recall*100:.2f} %')
print(f'F1             : {f1*100:.2f} %')
print(f'ROC-AUC        : {auc:.4f}')
print('\nConfusion matrix (rows=actual, cols=pred):')
print(confusion_matrix(y_test, y_pred))
print('\n', classification_report(y_test, y_pred, target_names=['normal', 'stress_event'], digits=3))

In [ ]:
# --- Feature importance (top 10) ---
imp = pd.DataFrame({'feature': FEATURES, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
print(imp.head(10).to_string(index=False))

plt.figure(figsize=(8, 5))
top = imp.head(15).iloc[::-1]
plt.barh(top['feature'], top['importance'], color='#1f77b4')
plt.title('Top 15 feature importances - Random Forest stress classifier')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

In [ ]:
# --- Persist artifacts for the Grid Monitor agent ---
os.makedirs('models', exist_ok=True)

PKL_PATH      = 'models/random_forest.pkl'
FEATURES_PATH = 'models/rf_feature_names.json'
CONFIG_PATH   = 'models/rf_config.json'

joblib.dump(rf, PKL_PATH)
with open(FEATURES_PATH, 'w') as fh:
    json.dump(FEATURES, fh, indent=2)

config = {
    'trained_at'      : pd.Timestamp.utcnow().isoformat(),
    'n_features'      : len(FEATURES),
    'n_estimators'    : 300,
    'max_depth'       : 15,
    'total_rows'      : int(len(df)),
    'train_rows'      : int(len(X_train)),
    'test_rows'       : int(len(X_test)),
    'target'          : TARGET,
    'class_weight'    : 'balanced',
    'test_metrics'    : {
        'accuracy' : float(accuracy),
        'precision': float(precision),
        'recall'   : float(recall),
        'f1'       : float(f1),
        'auc_roc'  : float(auc),
    },
    'top_features'    : imp.head(10).to_dict(orient='records'),
}
with open(CONFIG_PATH, 'w') as fh:
    json.dump(config, fh, indent=2, default=float)

print('Saved:')
print('  ', PKL_PATH, f'({os.path.getsize(PKL_PATH)/1e6:.1f} MB)')
print('  ', FEATURES_PATH)
print('  ', CONFIG_PATH)

if IN_COLAB:
    from google.colab import files
    for path in [PKL_PATH, FEATURES_PATH, CONFIG_PATH]:
        files.download(path)